# 파이썬의 변수 : 객체에 붙은 '이름표(Name/Binding)'이며, **= 은 참조 바인딩으로, 값을 복사하는 게 아니라, 같은 객체에 또 다른 이름을 붙이는 것이다.**

- 파이썬에서 `a = [1, 2, 3]` 은 리스트라는 상자에 `a` 라는 포스트잇을 붙이는 것과 같습니다.

* **참조 바인딩 (`=`)**: `b = a` 라고 하면, 기존 상자에 `b` 라는 포스트잇을 하나 더 붙이는 것입니다.

>### **Mutable(가변) vs Immutable(불변)**

>> **Immutable**
- int, float, str, tuple
- b= a라고 하면, 마찬가지로 b가 a와 동일한 주소를 가리킴.
- 다만, 값을 바꾸려 하면 아예 **새로운 상자** 를 만들어서 이름표를 옮겨 붙입니다. 그래서 서로 영향을 주지 않습니다.

>> **Mutable**
* 상자 **내부의 내용물** 을 바꾸는 것이 가능합니다. 이름표가 여러 개 붙어 있다면, 어떤 이름을 부르든 바뀐 내용물이 보입니다.


---

# **복사의 3단계**

### ① 단순 할당 (Reference Binding)

* **코드**: `b = a`
* **상태**: 하나의 객체를 공유. --> 'b` 를 수정하면 `a` 도 무조건 바뀜.

### ② 얕은 복사 (Shallow Copy)
- 컨테이너 객체(리스트, 딕셔너리오 같이 다른 객체들을 포함하고 관리하는 객체) 자체만 새로 생성하고, 그 내부에 포함된 요소(element)들에 대해서는 기존 객체에 대한 참조(reference)를 그대로 공유하는 복사 방식이다.
- **Shallow copy는 “내부 객체가 Immutable일 때만” 안전하다.**

In [ ]:
b = a.copy()
b = a[:]
b = list(a)

✔ 메모리 관점
- 최상위 컨테이너 객체(1-depth): 새로 할당
- 내부 요소(2-depth 이상): 기존 객체 주소 공유

In [ ]:
a = [[1,2],[3,4]]
b = a[:]

In [ ]:
b[0][0] = 9
a

[[9, 2], [3, 4]]

In [ ]:
"""
a ──► list ──► [ list1 , list2 ]
b ──► list ──┘   ↑      ↑
                     공유    공유
"""
# 그렇기에, b[0][0] = 9 와 같이 b를 수정하면, a도 수정됨


### ③ 깊은 복사 (Deep Copy)

* 컨테이너 객체뿐만 아니라, 그 내부에 중첩된 모든 객체를 재귀적으로 새로 생성하여
원본 객체와 완전히 독립적인 객체 그래프(object graph)를 만드는 복사 방식이다.


In [ ]:
# 정확한 Deep Copy
import copy
b = copy.deepcopy(a)

✔ 메모리 관점 동작

- 모든 depth에서 새 객체 생성

- 원본과 참조 관계 완전히 분리

In [ ]:
a ──► list ──► list ──► value
b ──► list ──► list ──► value

> 다만, 객체 수에 비례한 O(N) ~ O(N×depth) 시간으로 인해 시간 초과 위험이 있어, 보통 완전한 deepcopy는 잘 사용하지 않는다.

>> ## 따라서, 가변 객체(Mutable Object)는 명시적으로 복사한다.
- 파이썬에서 가변 객체(mutable object) 는
단순 참조 바인딩(=)만으로는 객체의 독립성이 보장되지 않으므로, 의도한 변경 범위(depth)에 맞추어 명시적으로 복사해야 한다.

- 이 방식은 모든 중첩 객체를 재귀적으로 복제하는 완전한 Deep Copy는 아니지만,  **실제로 변경이 발생하는 깊이까지만 객체를 분리하는 Depth-limited Deep Copy(부분적 깊은 복사)** 로 볼 수 있다. 이러한 복사 방식을 통해  문제에서 요구하는 수준의 객체 독립성은 충분히 보장하며,   불필요한 연산 비용을 피할 수 있다.

🔹 1차원 얕은 복사 (Shallow Copy)

In [ ]:
b = a[:]

"""
✔ 내부 요소가 Immutable (int, str, tuple 등)하다면, 안전하게 복사됨
✔ 최상위 리스트 객체(1-depth)만 새로 생성 & 내부 요소는 그대로 참조

a ──► list ──► value
b ──► list ──┘
"""

🔹 2차원 전용 deep copy

In [ ]:
b = [row[:] for row in a]
"""
1-depth: 외부 리스트 새로 생성
2-depth: 내부 리스트 각각 새로 생성
3-depth 이상: 참조 유지

a ──► list ──► list ──► value
b ──► list ──► list ──┘
"""

🔹 3차원 깊은 복사

In [ ]:
b = [[col[:] for col in row] for row in a]

"""
1-depth: 외부 리스트 복사
2-depth: 중간 리스트 복사
3-depth: 내부 리스트 복사

a ──► list ──► list ──► list ──► value
b ──► list ──► list ──► list ──┘
"""

> 이전 상태의 지도 정보를 보존해야하는 경우

In [ ]:
import copy

# 원본 지도 (2차원 리스트)
original_map = [
    [0, 1, 0],
    [0, 0, 2],
    [1, 0, 0]
]

# 1. 잘못된 복사 (참조만 복사)
# tmp_map = original_map  <- 절대 금지!

# 2. 얕은 복사 (안쪽 리스트가 공유됨)
# tmp_map = original_map.copy() <- 위험!

# 3. 권장하는 방식 (2차원 깊은 복사 효과 + 속도)
tmp_map = [row[:] for row in original_map]

# 이제 tmp_map을 마음껏 수정해도 original_map은 안전합니다.
tmp_map[0][0] = 9
print(original_map[0][0]) # 결과: 0 (안전하게 보존됨)

# **List**
- 순서가 있고 수정이 가능한(Mutable) 자료구조

In [ ]:
# list의 생성
arr1 = [1,2,3]
arr2 = list()

- append(x): 리스트 끝에 $x$ 를 추가 (시간 복잡도: $O(1)$).
- insert(i, x): $i$ 번 인덱스에 $x$ 를 삽입 (시간 복잡도: $O(N)$).
- pop(i): $i$ 번 인덱스의 요소를 제거하고 반환 (기본값은 마지막 요소, $O(1)$).
- remove(x): 값이 $x$ 인 첫 번째 요소를 삭제 ($O(N)$).

- sort(): 원본 리스트를 오름차순으로 정렬.

- sort(reverse=True): 내림차순 정렬.

In [ ]:
a = [3, -1, 5, -10]
a.sort(key=lambda x: abs(x))
# key에 들어가는 건 "함수"

> 리스트 뒤집기와 슬라이싱

- arr[::-1]: 리스트 전체를 뒤집은 새 리스트를 반환합니다.
- arr[k:] + arr[:k]: 리스트를 $k$ 만큼 회전시킬 때 유용합니다.

> key를 활용한 정렬 (Sorting)
- 다중 조건 정렬: arr.sort(key=lambda x: (x[0], -x[1]))

- 첫 번째 인자를 기준으로 오름차순, 첫 번째가 같으면 두 번째 인자로 내복차순 정렬합니다.

- 길이 기준 정렬: arr.sort(key= lambda x: len(x))

> 특정 값의 개수와 존재 여부
- arr.count(x): 리스트 내 $x$ 의 개수를 셉니다 ($O(N)$).
- if x in arr:: 리스트 안에 $x$ 가 있는지 확인합니다 ($O(N)$).
- **Tip: 존재 여부만 계속 확인해야 한다면 리스트 대신 set 을 쓰는 것이 훨씬 빠릅니다 ($O(1)$).**


> ① 중첩 리스트 펼치기 (Flattening) : 2차원 리스트를 1차원으로 합칠 때 유용

In [ ]:
flat = [x for row in matrix for x in row]

> List에서 값이 없는 index에 값을 새로 넣고자 할 때는 append나 extend를 사용해야함

> ### **List Comprehension**
- ### [표현식 for 항목 in 반복가능객체 if 조건문]

In [ ]:
# 2차원 배열 초기화
# 5행 10열의 0으로 채워진 격자
grid = [[0] * 10 for _ in range(5)]
# 격자(Grid) 문제를 풀 때 필수적으로 사용됩니다.

"""
array = [[0] * 3] * 3 처럼 쓰면,
세 행이 모두 같은 메모리 주소를 가리키게 된다.
"""

# 필터링 기능 : 0~9 중 짝수만 리스트로 만들기
evens = [x for x in range(10) if x % 2 == 0]
# 결과: [0, 2, 4, 6, 8]

# **Tuple**
- 순서가 있지만 수정이 불가능한(Immutable) 자료구조
- 한 번 생성된 요소는 변경하거나 삭제할 수 없습니다.
- 리스트에 비해 상대적으로 메모리를 적게 사용합니다.

>> - 코딩 테스트에서는 그래프 알고리즘(Dijkstra 등) 에서 (거리, 노드) 처럼 변경되지 않는 데이터 쌍을 묶어 관리할 때 주로 사용
>> - 딕셔너리의 키 로 사용할 수 있다

In [ ]:
# 생성
t1 = (1, 2, 3)
t2 = tuple()

# 원소가 1개뿐인 tuple 만들기 : ,를 붙여줘야 한다.
# 수학 연산에서의 우선순위 괄호와 구별되어야
t3 = (1, )

In [ ]:
list_to_tuple = tuple([1, 2, 3]) # (1, 2, 3)
str_to_tuple = tuple("ABC")      # ('A', 'B', 'C')

> ① 튜플 언패킹 (Unpacking)

In [ ]:
def get_pos():
    return (10, 20)

x, y = get_pos() # 자동으로 튜플의 값이 분리되어 들어감

② 우선순위 큐(Priority Queue) 와의 조합

- heapq.heappush(hq, (priority, value)) : 우선순위가 낮은 순서대로 정렬됩니다

③ 딕셔너리의 키 (Key) 로 사용
- 리스트는 내용이 바뀔 수 있어 딕셔너리의 키가 될 수 없지만, 튜플은 가능

In [ ]:
visited = {(0, 0): True}
# 좌표를 키로 사용하여 방문 여부를 체크할 때 매우 유용

>> 함수가 여러 값을 반환할 때에는, 자동으로 튜플이 생성된다.

In [ ]:
def get_data():
    return 1, 2 # 사실 (1, 2)라는 튜플을 반환하는 것

a, b = get_data() # 튜플 언패킹으로 값 받기

# **Set**
- **중복을 허용하지 않고 순서가 없는 자료구조**

- set에서는 indexing 불가
- 생성: s = set() (빈 중복 제거 집합), s = {1, 2, 3}

> - hash 값(hash value) 이란? 👉 어떤 값을 “고정된 정수”로 변환한 결과
>> - set은 해시 테이블 기반 자료구조이기 때문에 원소의 해시값이 저장 이후에도 변하지 않아야 한다. mutable 객체는 값이 변경될 수 있어 해시값이 바뀔 수 있고, 그러면 set 내부 위치와 실제 값이 불일치해 자료구조가 깨집니다. 그래서 Python에서는 mutable 객체를 set 원소로 허용하지 않습니다.

> - add(x)
> - remove(x)    # 없으면 에러
> - discard(x)   # 없어도 안전


### <핵심 개념 및 문법>

- ### 생성: s = set() (빈 중복 제거 집합), s = {1, 2, 3}

>> #### **이때, dict 와 set은 모두 { }으로 감싸지지만, 빈 dictionary를 만들 때에는 a = { } 로 하거나 a = dict()를 쓰지만, 비어있는 set은 항상 set()으로 만들어야 한다. 추가로 {2} 는 set이고, {2: 3}은 dictonary이다.**

- 추가/삭제: s.add(val), s.remove(val) (없으면 에러), s.discard(val) (없어도 안전)


In [ ]:
print(hash("apple"))   # 예: 3456234987623
print(hash(10))        # 예: 10
print(hash((1, 2)))    # 예: -3550055125485641917
print(hash("apple"))

# 값이 같으면 항상 같은 hash 값
# 값이 다르면 보통 다른 hash 값

-1626438873212484803
10
-3550055125485641917
-1626438873212484803


① 중복 제거
- 리스트를 set으로 변환했다가 다시 리스트로 만들면 중복이 사라집니다.
- list(set(my_list))

In [ ]:
lst = [1, 2, 2, 3, 3]
unique = list(set(lst))

② 존재 여부 확인 in
- 리스트에서 x in list는 $O(N)$ 이지만, 세트에서 x in set은 $O(1)$ 입니다.
- 데이터 양이 많을 때 탐색

③ 교집합 / 차집합 / 합집합

- 두 배열의 공통 원소 개수
- 한쪽에만 있는 값 찾기

> 집합 연산:

> - 합집합: a | b 또는 a.union(b)

> - 교집합: a & b 또는 a.intersection(b)

> - 차집합: a - b 또는 a.difference(b)

# **Dictionary**

- Key → Value 매핑
- 키(Key)를 통해 값(Value)을 순식간에 찾아내는 자료구조입니다.
- “빈도, 매핑, 상태 관리”에 주로 사용됨
> - 접근 및 추가: d["key"] = value
> - 안전한 접근: d.get("key", default) (키가 없으면 default 반환, 에러 방지)

In [ ]:
# 생성
# 방법 1: 중괄호 사용
empty_dict = {}

# 방법 2: dict() 생성자 사용
empty_dict = dict()


d = {'a': 1, 'b': 2}

① 빈도 세기 (최빈값 문제)
- dict.get(key, default)

In [ ]:
count = {}
arr = [2,3,5,5]
for x in arr:
    count[x] = count.get(x, 0) + 1
    # 딕셔너리 count에서 key x가 있으면 그 값, 없으면 0을 반환

print(count)

count = {'a': 2}
print(count.get('a', 0))  # 2
print(count.get('b', 0))  # 0

{2: 1, 3: 1, 5: 2}
2
0


② 상태 저장 / 누적 관리
- 누적 합
- 등장 횟수
- 점수 테이블
- 방문 여부

In [ ]:
score[name] += point

③ 매핑 문제 (치환, 변환)

In [ ]:
mapping = {'A': 1, 'B': 2}
result = mapping[x]

🔹 순회

In [ ]:
for k in d.keys(): (키 순회)

for v in d.values(): (값 순회)

for k, v in d.items(): (키와 값 동시에 순회)

> ### Dictionary 합치기 : update() 메서드 (기존 딕셔너리 변경)

- dict1 에 dict2 의 내용을 덮어씁니다. dict1 자체가 수정됩니다.

In [ ]:
dict1 = {"TV": 2000000, "냉장고": 1500000}
dict2 = {"노트북": 1200000, "세탁기": 1000000}

dict1.update(dict2)
print(dict1)
# 결과: {'TV': 2000000, '냉장고': 1500000, '노트북': 1200000, '세탁기': 1000000}

>>  key가 같을 때, "더 큰 값"을 유지하고 싶다면?

In [ ]:
dict1 = {"TV": 2000000, "냉장고": 1500000, "세탁기": 800000}
dict2 = {"TV": 2500000, "냉장고": 1200000, "스타일러": 1000000}

# dict1을 복사해서 시작
merged_dict = dict1.copy()

for key, value in dict2.items():
    # 키가 이미 존재한다면 더 큰 값을 선택, 없다면 새 키 추가
    if key in merged_dict:
        merged_dict[key] = max(merged_dict[key], value)
    else:
        merged_dict[key] = value

print(merged_dict)
# 결과: {'TV': 2500000, '냉장고': 1500000, '세탁기': 800000, '스타일러': 1000000}

> ### **딕셔너리 정렬 (sorted() 와 lambda)**
- 딕셔너리를 **값이 큰 순서대로 정렬하라**는 조건이 붙으면 리스트 형태로 변환하여 정렬해야 함.

>> **딕셔너리 정렬 과정**
- items() 함수를 사용하여 딕셔너리를 (Key, Value) 쌍의 튜플들로 이루어진 객체로 바꿉니다.

- 이 객체를 sorted() 함수를 사용하여 원하는 기준(값)으로 정렬합니다.

In [ ]:
score_dict = {'철수': 85, '영희': 95, '민수': 70}
score_dict.items()

dict_items([('철수', 85), ('영희', 95), ('민수', 70)])

In [ ]:
list(score_dict.items())

[('철수', 85), ('영희', 95), ('민수', 70)]

> 키(Key)를 기준으로 정렬할 때 : 딕셔너리의 키를 가나다순이나 알파벳순으로 나열하고 싶을 때 사용

In [ ]:
sorted_by_keys = sorted(score_dict.items())
sorted_by_keys

[('민수', 70), ('영희', 95), ('철수', 85)]

In [ ]:
lst_sort = sorted(list(score_dict.items()))
lst_sort

[('민수', 70), ('영희', 95), ('철수', 85)]

> 값(Value)을 기준으로 정렬할 때 : lambda 를 활용해 기준을 정해준다.

In [ ]:
score_dict = {'철수': 85, '영희': 95, '민수': 70}

# d.items()는 (key, value) 쌍을 꺼내줍니다.
# x[1]은 value를 의미하므로, 값을 기준으로 오름차순 정렬
# 1. 값이 큰 순서(내림차순)로 정렬하기

sorted_scores = sorted(score_dict.items(), key=lambda x: x[1], reverse= False)
# dict_prd.items() : dict_items 라는 이름의 객체로, just 보여주기 위한 객체
# sorted() : 정렬된 요소들을 새로운 list 객체에 담아주는 역할

print(sorted_scores)
# 결과: [('영희', 95), ('철수', 85), ('민수', 70)] -> 리스트 안에 튜플이 들어간 형태

[('민수', 70), ('철수', 85), ('영희', 95)]


> **다중 조건 정렬**

- 복합 조건에서는 `lambda` 에 튜플을 넣어주면 됩니다.

In [ ]:
d = {'apple': 5, 'banana': 5, 'cherry': 8}

# 1순위: 값 내림차순 (-x[1]), 2순위: 키 오름차순 (x[0])
sorted_res = sorted(d.items(), key=lambda x: (-x[1], x[0]))

print(sorted_res)
# 결과: [('cherry', 8), ('apple', 5), ('banana', 5)]

[('cherry', 8), ('apple', 5), ('banana', 5)]


# **문자열 string**

- Python의 문자열은 한 번 생성하면 그 안의 글자를 직접 바꿀 수 없습니다 -> s[0] = 'A' 와 같은 연산은 에러 발생

- 수정을 원하면 슬라이싱을 이용해 새로 만들거나, 리스트(List) 로 변환 후 수정하고 다시 문자열로 합쳐야 합니다.


> ① 분리 및 결합

- split(): 특정 구분자를 기준으로 문자열을 나눠 리스트로 만듭니다. (공백 기준이 기본) + **split()은 나누는 횟수를 제한할 수 있다**

- join(): 리스트의 문자열들을 하나로 합칩니다. "".join(list) 형태를 가장 많이 씁니다.

In [ ]:
# split(sep, maxsplit)
url = input().strip()

# 1. protocol 분리
protocol, rest = url.split("://", 1)

# 2. host 와 others 분리
host, others = rest.split("/", 1)

print(f"protocol: {protocol}")
print(f"host: {host}")
print(f"others: {others}")

http://www.example.com/test?p=1&q=2
protocol: http
host: www.example.com
others: test?p=1&q=2


In [ ]:
# join
"구분자".join(iterable)

In [ ]:
lst = ['a', 'b', 'c']
result = "".join(lst)
print(result)

# join을 하려 하는데, 문자가 아닌 게 섞여있다면?
lst = ['a', 'b', 3]
"".join(map(str, lst))

abc


>> 문자열 뒤집기

In [ ]:
s = "hello"
result = "".join(reversed(s))

>> 조건에 맞는 문자만 남기기

In [ ]:
s = "a1b2c3"
result = "".join(c for c in s if c.isalpha())

In [ ]:
"문자열은 immutable -> 매번 새 문자열을 생성하면 시간복잡도가 O(N^2)"
s = ""
for c in lst:
    s += c

> ② 탐색 및 확인

- find() / index(): 특정 문자의 위치를 찾습니다. (find는 없으면 -1 반환, index는 에러 발생)

- startswith() / endswith(): 특정 문자열로 시작하거나 끝나는지 확인 (조건문에서 유용).

- in 연산자: "abc" in "abcdef" 처럼 부분 문자열 포함 여부를 판단합니다.

In [ ]:
# find( )

s = "abcdef"

print(s.find('c'))   # 2
print(s.find('z'))   # -1  : find는 없으면 -1 반환

# error가 나지 않기에, 조건문에서 주로 많이 사용
if s.find('d') != -1:
    print("d가 존재함")

2
-1
d가 존재함


In [ ]:
# index

s = "abcdef"

print(s.index('c'))  # 2
print(s.index('z'))  # ValueError 발생

# 예외 처
try:
    pos = s.index('a')
    print(pos)
except ValueError:
    print("문자가 없음")

In [ ]:
# startswith() / endswith()

filename = "report_2026.pdf"

print(filename.startswith("report"))   # True
print(filename.startswith("data"))     # False

In [ ]:
# 튜플을 넣을 수도 있다
filename = "photo.jpeg"

if filename.endswith((".png", ".jpg", ".jpeg")):
    print("이미지 파일")

In [ ]:
# in 연산자
email = "user@gmail.com"

if "@" in email:
    print("올바른 이메일 형식")

올바른 이메일 형식


> ③ 수정 및 변환

- replace(old, new): 모든 old를 new로 바꿉니다.

- strip(), lstrip(), rstrip(): 양쪽 또는 한쪽의 공백 or 특정 문자를 제거합니다.

- lower(), upper(): 대소문자 변환.

In [ ]:
# replace(old, new)
s = "a-b-c-d"
s = s.replace("-", "_")

In [ ]:
# strip() / lstrip() / rstrip()
s = "   hello world   "

s.strip()    # "hello world"
s.lstrip()   # "hello world   "
s.rstrip()   # "   hello world"

# strip() 안에 특정 문자를 넣으면,
# 그 문자들이 문자열의 양 끝에 붙어 있을 때만 지워줍니다.
s = "###data###"
s.strip("#")   # "data"

'   hello world'

> ④ 성분 판별

- isdigit(): 숫자로만 이루어져 있는가? (입력값이 숫자인지 판단할 때 필수)

- isalpha(): 알파벳으로만 이루어져 있는가?

- isalnum(): 알파벳 또는 숫자로만 이루어져 있는가?

In [ ]:
"123".isdigit()     # True
"12.3".isdigit()   # False
"-123".isdigit()   # False

False

In [ ]:
"abc".isalpha()     # True
"ABC".isalpha()     # True
"abc1".isalpha()   # False

In [ ]:
"abc123".isalnum()   # True
"abc_123".isalnum()  # False

>> 아스키 코드 (ASCII) 변환
- 문자열을 숫자로 바꿔서 연산해야 할 때가 많습니다 (예: 알파벳 'a'를 0으로, 'b'를 1로 매핑).

- ord(문자): 문자를 아스키 코드로 변환 (예: ord('A') 는 65)

- chr(숫자): 아스키 코드를 문자로 변환 (예: chr(65) 는 'A')

- 활용 팁: ord(s) - ord('a') 를 하면 알파벳을 0~25 사이의 인덱스로 바꿀 수 있습니다.

>> 문자열을 알파벳 순으로 정렬

In [ ]:
s = "banana"
sorted_s = sorted(s) # ['a', 'a', 'a', 'b', 'n', 'n'] 리스트 반환
new_s = "".join(sorted_s) # "aaabnn"
new_s

'aaabnn'

>> “문자열로 붙어 들어온 격자 입력을
즉시 계산 가능한 2차원 숫자 지도 형태로 만들기

In [ ]:
grid_2d = [list(map(int, list(input()))) for _ in range(N)]

"""
5
00110
01001
11100
00011
10101

입력이 이렇게 들어오는 경우, split을 쓰면 안 된다.
"""

>> 짝수 인덱스를 가진 문자들을 출력하기

In [1]:
s = "H1e2l3l4o5w6o7r8l9d"
print(s[::2])

result = [ch for i, ch in enumerate(s) if i % 2 == 0]
print(''.join(result))

print(''.join(s[i] for i in range(0, len(s), 2)))

Helloworld


# class


In [ ]:
class A:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def move(self):
        self.x += 1

In [ ]:
# heap / 우선순위 큐에서 비교
import heapq

class Node:
    def __init__(self, cost, x):
        self.cost = cost
        self.x = x

    def __lt__(self, other):
        return self.cost < other.cost